# Text Vectorizer: BoW, N-gram, TF-IDF, dan Word Embedding

Notebook ini menjalankan ulang dan memverifikasi seluruh contoh kode dari materi *Text Vectorizer*, termasuk revisi berdasarkan notebook praktik asli (bukan cuma slide) yang mengungkap teknik Word Embedding yang benar-benar dipakai di kelas.

**Cakupan:**
1. Bag of Words (BoW)
2. N-gram
3. TF-IDF
4. Word Embedding (spaCy) — dengan audit temuan penting
5. Bonus: Word2Vec (gensim) untuk perbandingan pendekatan


## 0. Instalasi & Import Library

In [ ]:
!pip install -q scikit-learn gensim spacy Sastrawi
!python -m spacy download en_core_web_sm -q

In [2]:
import re
import numpy as np
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
from spacy.lang.id import Indonesian
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

np.set_printoptions(precision=4, suppress=True)
print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


## 1. Bag of Words (BoW)

BoW mengubah teks menjadi vektor berdasarkan **frekuensi kemunculan kata**, tanpa memperhatikan urutan kata.

Contoh ini pakai 3 dokumen, salah satunya dengan kata berulang ("learning" muncul 2x), supaya kelihatan jelas bagaimana BoW menghitung frekuensi — bukan cuma ada/tidak adanya kata.

In [3]:
dokumen = [
    "saya suka machine learning learning",
    "machine learning adalah masa depan",
    "natural language processing adalah masa depan",
]

bow_vectorizer = CountVectorizer()
bow_vektor = bow_vectorizer.fit_transform(dokumen)

print("Vocabulary:", list(bow_vectorizer.get_feature_names_out()))
print("Matriks BoW:")
print(bow_vektor.toarray())

Vocabulary: ['adalah', 'depan', 'language', 'learning', 'machine', 'masa', 'natural', 'processing', 'saya', 'suka']
Matriks BoW:
[[0 0 0 2 1 0 0 0 1 1]
 [1 1 0 1 1 1 0 0 0 0]
 [1 1 1 0 0 1 1 1 0 0]]


**Cara baca:** kolom `"learning"` di baris pertama nilainya **2** — karena kata itu memang muncul dua kali di kalimat `"saya suka machine learning learning"`. Ini yang membedakan BoW dari sekadar "ada/tidak ada": BoW betul-betul menghitung frekuensi, bukan cuma kehadiran kata.

Vectorizer yang sudah di-fit juga bisa dipakai untuk mengubah kalimat baru — termasuk kalimat yang mengandung kata di luar vocabulary:

In [4]:
print(bow_vectorizer.transform(["saya suka machine"]).toarray())
print(bow_vectorizer.transform(["saya suka machine apple"]).toarray())

[[0 0 0 0 1 0 0 0 1 1]]
[[0 0 0 0 1 0 0 0 1 1]]


Perhatikan: dua baris di atas hasilnya **identik**. Kata `"apple"` sama sekali tidak ada di vocabulary, jadi diabaikan begitu saja — bukan error, tapi juga bukan sesuatu yang otomatis "ketahuan" oleh vectorizer. Ini konsekuensi wajar dari cara kerja BoW yang perlu disadari saat dipakai di data production.

## 2. N-gram

BoW dan TF-IDF standar memecah teks jadi kata tunggal (*unigram*). N-gram memecahnya jadi kombinasi **beberapa kata berurutan**, supaya frasa seperti "machine learning" tetap dianggap satu unit, bukan dua kata terpisah.

In [5]:
bigram_vectorizer = CountVectorizer(analyzer='word', ngram_range=(2, 2))
X_bigram = bigram_vectorizer.fit_transform(dokumen)

print("Vocabulary bigram:", list(bigram_vectorizer.get_feature_names_out()))
print(X_bigram.toarray())

Vocabulary bigram: ['adalah masa', 'language processing', 'learning adalah', 'learning learning', 'machine learning', 'masa depan', 'natural language', 'processing adalah', 'saya suka', 'suka machine']
[[0 0 0 1 1 0 0 0 1 1]
 [1 0 1 0 1 1 0 0 0 0]
 [1 1 0 0 0 1 1 1 0 0]]


Sekarang `"machine learning"` jadi satu fitur sendiri, terpisah dari `"learning adalah"` atau `"suka machine"`. Trade-off-nya: vocabulary jadi lebih besar dan lebih jarang muncul berulang (sparse), tapi frasa yang punya makna gabungan (seperti nama entitas atau idiom) lebih terjaga.

## 3. TF-IDF (Term Frequency–Inverse Document Frequency)

TF-IDF memberi bobot lebih rendah untuk kata yang terlalu umum (muncul di banyak dokumen) dan bobot lebih tinggi untuk kata yang unik.

In [6]:
documents = ["saya suka machine learning", "machine learning adalah masa depan"]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectors = tfidf_vectorizer.fit_transform(documents)

print("Vocabulary:", list(tfidf_vectorizer.get_feature_names_out()))
print("Matriks TF-IDF:")
print(tfidf_vectors.toarray())

Vocabulary: ['adalah', 'depan', 'learning', 'machine', 'masa', 'saya', 'suka']
Matriks TF-IDF:
[[0.     0.     0.4099 0.4099 0.     0.5762 0.5762]
 [0.4992 0.4992 0.3552 0.3552 0.4992 0.     0.    ]]


Kata `"machine"` dan `"learning"` — yang muncul di kedua dokumen — mendapat bobot lebih rendah (≈0.36–0.41) dibanding kata yang cuma muncul di satu dokumen seperti `"saya"`/`"suka"` atau `"adalah"`/`"masa"`/`"depan"` (≈0.50–0.58). TF-IDF otomatis "tahu" kata mana yang lebih membedakan satu dokumen dari yang lain — sesuatu yang tidak bisa dilakukan BoW biasa.

## 4. Word Embedding dengan spaCy

Ini bagian yang direvisi setelah menemukan notebook praktik asli dari kelas: teknik yang benar-benar dipakai adalah **spaCy**, bukan Word2Vec/gensim seperti contoh ilustratif di slide (lihat bagian 5 untuk perbandingannya).

In [7]:
nlp = spacy.load("en_core_web_sm")

vektor_machine = nlp("machine").vector
vektor_natural = nlp("natural language processing").vector

print("Dimensi vektor 'machine':", vektor_machine.shape)
print("5 nilai pertama:", vektor_machine[:5])

Dimensi vektor 'machine': (96,)
5 nilai pertama: [-0.7506 -0.5765  0.6435  0.3477  0.4501]


> ⚠️ **Temuan audit paling penting di notebook ini:** `en_core_web_sm` adalah model **kecil** dari spaCy, dan model kecil **tidak dilengkapi word vector asli sama sekali**. Mari kita buktikan langsung:

In [8]:
print("Jumlah real word vector di vocabulary:", len(nlp.vocab.vectors))

import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    similarity = nlp("machine").similarity(nlp("natural"))
    if w:
        print("\nWarning dari spaCy sendiri:")
        print(str(w[0].message))

print(f"\nSimilarity 'machine' vs 'natural': {similarity:.4f}")

Jumlah real word vector di vocabulary: 0

Warning dari spaCy sendiri:
[W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.

Similarity 'machine' vs 'natural': 0.3831


`len(nlp.vocab.vectors)` menghasilkan **0** — artinya `en_core_web_sm` tidak menyimpan satu pun word vector asli. Angka `.vector` yang ditampilkan sebelumnya bukan representasi makna kata seperti Word2Vec/GloVe, melainkan cuma tensor internal dari komponen tagger/parser/NER milik model itu. spaCy sendiri sampai memunculkan **warning eksplisit** kalau method `.similarity()` dipakai dalam kondisi ini, karena hasilnya "may not give useful similarity judgements".

**Kenapa ini penting:** kalau notebook aslinya menampilkan angka vektor dan similarity tanpa disertai warning ini, pembaca bisa mengira itu representasi makna yang valid — padahal secara teknis itu keliru. Untuk word vector asli, spaCy punya model yang lebih besar (`en_core_web_md` atau `en_core_web_lg`) yang memang dilengkapi pretrained vectors; `en_core_web_sm` secara dokumentasi resmi memang tidak menyertakannya, demi ukuran file yang lebih kecil.

### 4.1 Bug Kedua: Lemmatizer Bahasa Indonesia yang Selalu Kosong

Notebook asli juga mencoba tokenisasi + lemmatisasi Bahasa Indonesia pakai `spacy.lang.id.Indonesian()`. Mari kita coba jalankan persis seperti aslinya:

In [9]:
nlp_id = Indonesian()
nlp_id.Defaults.stop_words.update(['nya', 'yg', 'aja', 'deh', 'ny', 'dr', 'sy', 'ya', 'klo', 'sdh',
                                     'udah', 'sampe', 'dah', 'tp', 'ga', 'gk', 'sih', 'gak', 'tdk', 'e', 'dgn', 'sm'])

def tokenizer_original(text):
    return [token.lemma_.lower() for token in nlp_id(text) if not token.is_stop and not token.is_punct]

contoh = "PDIP resmi mencalonkan Gubernur DKI Jakarta, Jokowi sebagai calon presiden."
print("Pipeline components:", nlp_id.pipe_names)
print("Hasil tokenizer asli:", tokenizer_original(contoh))

Pipeline components: []
Hasil tokenizer asli: ['', '', '', '', '', '', '', '', '']


`Pipeline components: []` — kosong total. `Indonesian()` di spaCy itu **blank pipeline**, cuma py tokenizer doang, tanpa komponen lemmatizer apa pun. Makanya `token.lemma_` selalu mengembalikan string kosong, dan hasil akhirnya jadi daftar string kosong `['', '', ...]` — persis seperti yang muncul di notebook aslinya. Ini bukan salah cara menjalankan; ini keterbatasan nyata dari pipeline kosong tersebut.

**Perbaikan:** ganti `token.lemma_` dengan **Sastrawi** untuk stemming Bahasa Indonesia yang sebenarnya bekerja — sama seperti yang sudah dipakai di proyek sistem tanya-jawab FAQ sebelumnya.

In [10]:
stemmer = StemmerFactory().create_stemmer()

def tokenizer_fixed(text):
    text = re.sub(r'[^\w\s]', '', text.lower())
    doc = nlp_id(text)
    return [stemmer.stem(token.text) for token in doc if not token.is_stop]

print("Hasil tokenizer diperbaiki:", tokenizer_fixed(contoh))

Hasil tokenizer diperbaiki: ['pdip', 'resmi', 'calon', 'gubernur', 'dki', 'jakarta', 'jokowi', 'calon', 'presiden']


Sekarang hasilnya berupa kata-kata dasar yang sebenarnya, bukan string kosong.

## 5. Bonus: Word Embedding dengan Word2Vec (gensim)

Ini teknik yang muncul sebagai contoh ilustratif di slide materi. Disertakan di sini sebagai perbandingan pendekatan, bukan sebagai pengganti bagian 4.

In [11]:
sentences = [
    ["saya", "suka", "machine", "learning"],
    ["machine", "learning", "adalah", "masa", "depan"],
]

w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=1, seed=42)
vec_machine = w2v_model.wv["machine"]

print("Dimensi vektor 'machine':", vec_machine.shape)
print("Kata paling 'mirip' dengan 'machine':", w2v_model.wv.most_similar("machine", topn=3))

Dimensi vektor 'machine': (100,)
Kata paling 'mirip' dengan 'machine': [('learning', 0.09104208648204803), ('saya', 0.02587900310754776), ('adalah', 0.005806456319987774)]


**Catatan jujur yang sama seperti sebelumnya:** korpus di atas cuma 2 kalimat pendek, jadi similarity score-nya (semua di bawah 0.1) belum bermakna secara semantik — sama seperti masalah "zero vectors" di spaCy `en_core_web_sm`, cuma dengan penyebab yang beda. Word2Vec butuh korpus besar (jutaan kata) untuk hasil yang valid, sementara spaCy `en_core_web_sm` memang secara desain tidak menyertakan vector sama sekali.

**Kesimpulan gabungan:** dua model, dua cara gagal yang berbeda — tapi pelajarannya sama: **selalu cek apakah "vector" yang kita pakai benar-benar merepresentasikan makna**, jangan asumsikan begitu saja karena angkanya terlihat "ada".

## 6. Ringkasan

| Teknik | Kelebihan | Kekurangan / Hal yang Perlu Diwaspadai |
|---|---|---|
| Bag of Words | Sederhana, menangkap frekuensi kata | Abaikan urutan & makna; kata di luar vocabulary diabaikan diam-diam |
| N-gram | Menjaga frasa/kombinasi kata | Vocabulary membesar cepat, data jadi lebih sparse |
| TF-IDF | Kata umum otomatis dapat bobot lebih rendah | Masih abaikan konteks & urutan kata |
| Word Embedding (spaCy `en_core_web_sm`) | Praktis, satu baris kode | **Model kecil TIDAK punya word vector asli** — butuh `en_core_web_md`/`lg` untuk vector yang valid |
| Word2Vec (gensim) | Bisa melatih vector sendiri dari data spesifik domain | Butuh korpus besar; korpus kecil menghasilkan vector yang belum bermakna |

Dua bug nyata ditemukan di materi aslinya: (1) `en_core_web_sm` dipakai seolah punya word vector padahal tidak, dan (2) lemmatizer Bahasa Indonesia `spacy.lang.id.Indonesian()` selalu mengembalikan string kosong karena pipeline-nya kosong. Keduanya sudah diverifikasi ulang dan diberi solusi di notebook ini.


---

**Sumber materi:** rubythalib.ai AI Engineer Bootcamp — modul NLP (mentor: Muhammad Ikhwan Fathulloh).
**Catatan:** direvisi berdasarkan notebook praktik asli (bukan hanya slide); seluruh kode dijalankan ulang dan diverifikasi secara independen, termasuk dua bug yang ditemukan dan diperbaiki.

Proyek praktik penerapan TF-IDF (sistem tanya-jawab FAQ) dibahas terpisah di repo lain: `faq-search-engine-tfidf-sastrawi-arielshakaramiro`.
